# Toehold switch — two-input AND

Manual test harness for `engine.gates.toehold.ToeholdAndGate`. It inherits every
method from `ToeholdGate` except `generate_designs`, and sets `max_inputs = 2`.
Scientific methods print `pending Step 5` until the bodies land.

## The mechanism

An AND toehold opens only when **both** triggers are present. The usual shape is a
**serial stem**: trigger A opens an outer hairpin, exposing the toehold for trigger
B, which opens the inner hairpin holding the start codon. Either trigger alone
leaves the construct closed.

Two things this notebook is here to watch for once Step 5 lands:

* **Order matters** — A-outer/B-inner is a different construct from the reverse.
* **The intermediate (one-trigger) state is real** — if the start codon is already
  accessible with only the first trigger, the gate is an OR wearing an AND's shape.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

In [ ]:
host = fx.Host.ECOLI
gate = fx.toehold_and(host=host)
fx.describe_gate(gate)

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [ ]:
triggers = fx.sample_trigger_set(n_activators=2)   # two activators; may share a gene
constraints = fx.sample_constraints(max_switch_length=240)

for t in triggers.activators:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — implemented

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()` *(Step 5)*

Try it with a one-activator set too — a two-input family should reject arity 1
with a message written for the researcher.

In [ ]:
fx.attempt('is_compatible (2 activators)', lambda: gate.is_compatible(triggers, constraints))
one = fx.sample_trigger_set(n_activators=1)
fx.attempt('is_compatible (1 activator)', lambda: gate.is_compatible(one, constraints))

## `generate_designs()` *(Step 5)*

Expect both trigger orders to be generated, and the single-trigger states
evaluated explicitly.

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()` *(Step 5)*

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

These two are implemented today. `generate_designs()` is not, so
`fx.sample_design(...)` hands us a plausible `GateDesign` to call them on.

In [ ]:
design = fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Switching in real folding

Everything above runs against `fx.StubFoldEngine` — deterministic, fake, no
ViennaRNA. For genuine structure predictions, pass `real_fold=True` when you build
the gate (needs `import RNA` to work in this environment):

```python
gate = fx.toehold_and(host=host, real_fold=True)
folder = fx.fold_engine(real=True)
folder.mfe('GGGAAACCCUUUGGGAAACCC')   # -> FoldResult(structure, energy)
```